## Importar librerías y cargar el modelo y la base vectorial

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

# Rutas
VECTOR_STORE = Path("/home/jupyteruser/work/vector_store")
METADATA_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/metadatos")

# Cargar modelo de embeddings (el mismo que usamos)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Conectar a la base vectorial
client = chromadb.PersistentClient(path=str(VECTOR_STORE))
collection = client.get_collection("corpus_upeu")
print(f"Conexión exitosa. Documentos en colección: {collection.count()}")

/usr/local/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Conexión exitosa. Documentos en colección: 2488


## Definir el banco de preguntas de prueba

In [2]:
banco_preguntas = [
    {"id": 1, "consulta": "¿Cuáles son los requisitos para solicitar titulación?", "categoria_esperada": "C"},
    {"id": 2, "consulta": "¿Qué derechos tiene el estudiante unionista?", "categoria_esperada": "B"},
    {"id": 3, "consulta": "¿Cómo inscribir un proyecto de tesis?", "categoria_esperada": "C"},
    {"id": 4, "consulta": "¿Cuál es el proceso de convalidación de cursos?", "categoria_esperada": "D"},
    {"id": 5, "consulta": "¿Qué sanciones contempla el reglamento estudiantil?", "categoria_esperada": "B"},
    {"id": 6, "consulta": "¿Cuáles son los requisitos para reservar matrícula?", "categoria_esperada": "D"},
    {"id": 7, "consulta": "¿Cómo se solicita un certificado de estudios?", "categoria_esperada": "D"},
    {"id": 8, "consulta": "¿Cuál es el procedimiento para cambio de carrera?", "categoria_esperada": "D"},
    {"id": 9, "consulta": "¿Qué dice el estatuto sobre la autonomía universitaria?", "categoria_esperada": "A"},
    {"id": 10, "consulta": "¿Cómo presentar una queja o reclamo formal?", "categoria_esperada": "B"},
]

print(f"Banco de preguntas: {len(banco_preguntas)} consultas")

Banco de preguntas: 10 consultas


## Función para evaluar una consulta

In [3]:
def evaluar_consulta(consulta, n_resultados=3, umbral_distancia=0.8):
    """
    Realiza una búsqueda semántica y evalúa si los resultados son pertinentes.
    Retorna un diccionario con datos de la evaluación.
    """
    # Generar embedding de la consulta
    query_embedding = model.encode([consulta])[0].tolist()
    
    # Buscar en ChromaDB
    resultados = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_resultados,
        include=["documents", "metadatas", "distances"]
    )
    
    # Extraer datos
    docs = resultados['documents'][0]
    metas = resultados['metadatas'][0]
    dists = resultados['distances'][0]
    
    # Calcular métricas de pertinencia
    top1_dist = dists[0]
    top1_doc = metas[0]['documento'] if len(metas) > 0 else None
    top1_categoria = metas[0]['categoria'] if len(metas) > 0 else None
    top1_fragmento = docs[0][:200] if len(docs) > 0 else ""
    
    # Consideramos "cubierta" si la distancia del primer resultado es < umbral
    cubierta = top1_dist < umbral_distancia
    
    return {
        "consulta": consulta,
        "cubierta": cubierta,
        "distancia_top1": round(top1_dist, 4),
        "documento_top1": top1_doc,
        "categoria_top1": top1_categoria,
        "fragmento_top1": top1_fragmento,
        "n_resultados": len(docs),
        "distancias": [round(d, 4) for d in dists],
        "documentos_encontrados": [m['documento'] for m in metas]
    }

# Probar con una pregunta
ejemplo = evaluar_consulta(banco_preguntas[0]['consulta'])
print(f"Ejemplo:\nConsulta: {ejemplo['consulta']}\nCubierta: {ejemplo['cubierta']}\nDoc: {ejemplo['documento_top1']}\nDistancia: {ejemplo['distancia_top1']}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Ejemplo:
Consulta: ¿Cuáles son los requisitos para solicitar titulación?
Cubierta: True
Doc: REGLAMENTO DOCENCIA ORDINARIA v3.5
Distancia: 0.3037


## Evaluar todo el banco de preguntas

In [4]:
resultados_evaluacion = []

for pregunta in banco_preguntas:
    ev = evaluar_consulta(pregunta['consulta'])
    ev['id'] = pregunta['id']
    ev['categoria_esperada'] = pregunta['categoria_esperada']
    resultados_evaluacion.append(ev)
    print(f"Procesada consulta {pregunta['id']}: {pregunta['consulta'][:50]}... -> {'✓' if ev['cubierta'] else '✗'}")

# Crear DataFrame
df_resultados = pd.DataFrame(resultados_evaluacion)
print("\nResumen de cobertura:")
print(f"Consultas cubiertas: {df_resultados['cubierta'].sum()} de {len(df_resultados)}")
print(f"Tasa de cobertura: {df_resultados['cubierta'].mean()*100:.1f}%")

Procesada consulta 1: ¿Cuáles son los requisitos para solicitar titulaci... -> ✓
Procesada consulta 2: ¿Qué derechos tiene el estudiante unionista?... -> ✓
Procesada consulta 3: ¿Cómo inscribir un proyecto de tesis?... -> ✓
Procesada consulta 4: ¿Cuál es el proceso de convalidación de cursos?... -> ✓
Procesada consulta 5: ¿Qué sanciones contempla el reglamento estudiantil... -> ✓
Procesada consulta 6: ¿Cuáles son los requisitos para reservar matrícula... -> ✓
Procesada consulta 7: ¿Cómo se solicita un certificado de estudios?... -> ✓
Procesada consulta 8: ¿Cuál es el procedimiento para cambio de carrera?... -> ✓
Procesada consulta 9: ¿Qué dice el estatuto sobre la autonomía universit... -> ✓
Procesada consulta 10: ¿Cómo presentar una queja o reclamo formal?... -> ✓

Resumen de cobertura:
Consultas cubiertas: 10 de 10
Tasa de cobertura: 100.0%


## Análisis detallado por consulta

In [5]:
# Mostrar tabla de resultados
cols_mostrar = ['id', 'consulta', 'cubierta', 'distancia_top1', 'documento_top1', 'categoria_top1', 'categoria_esperada']
print("Resultados detallados:")
display(df_resultados[cols_mostrar])

# Identificar consultas no cubiertas
no_cubiertas = df_resultados[df_resultados['cubierta'] == False]
if len(no_cubiertas) > 0:
    print(f"\nConsultas con baja cobertura ({len(no_cubiertas)}):")
    for _, row in no_cubiertas.iterrows():
        print(f"\n  ID {row['id']}: {row['consulta']}")
        print(f"  Distancia: {row['distancia_top1']}  |  Documento: {row['documento_top1']}")
        print(f"  Fragmento: {row['fragmento_top1'][:150]}...")
else:
    print("\n¡Todas las consultas están cubiertas!")

Resultados detallados:


,id,consulta,cubierta,distancia_top1,documento_top1,categoria_top1,categoria_esperada
0,1,¿Cuáles son los requisitos para solicitar titu...,True,0.3037,REGLAMENTO DOCENCIA ORDINARIA v3.5,A,C
1,2,¿Qué derechos tiene el estudiante unionista?,True,0.3249,ESTATUTO 2024. 04-09-2024,A,B
2,3,¿Cómo inscribir un proyecto de tesis?,True,0.2674,REGLAMENTO INVESTIGACION UPeU 2025 V8,C,C
3,4,¿Cuál es el proceso de convalidación de cursos?,True,0.2905,REGLAMENTO DE ESTUDIOS V5_2025,D,D
4,5,¿Qué sanciones contempla el reglamento estudia...,True,0.1492,REGLAMENTO DE ESTUDIOS POSGRADO 2025,D,B
5,6,¿Cuáles son los requisitos para reservar matrí...,True,0.4277,REGLAMENTO DE ESTUDIOS POSGRADO 2025,D,D
6,7,¿Cómo se solicita un certificado de estudios?,True,0.2382,REGLAMENTO DE ESTUDIOS POSGRADO 2025,D,D
7,8,¿Cuál es el procedimiento para cambio de carrera?,True,0.4452,REGLAMENTO DE ESTUDIOS V5_2025,D,D
8,9,¿Qué dice el estatuto sobre la autonomía unive...,True,0.2712,REGLAMENTO GENERAL UPeU 2023,A,A
9,10,¿Cómo presentar una queja o reclamo formal?,True,0.4153,REGLAMENTO INTERNO DE TRABAJO Final 2020,A,B



¡Todas las consultas están cubiertas!


## Guardar informe de cobertura

In [6]:
# Guardar resultados completos
INFORME_COBERTURA = METADATA_FOLDER / "informe_cobertura.csv"
df_resultados.to_csv(INFORME_COBERTURA, index=False, encoding='utf-8')
print(f"Informe de cobertura guardado en {INFORME_COBERTURA}")

# Crear resumen estadístico
resumen = {
    "Total consultas": len(df_resultados),
    "Cubiertas": int(df_resultados['cubierta'].sum()),
    "No cubiertas": int((~df_resultados['cubierta']).sum()),
    "Tasa cobertura": round(df_resultados['cubierta'].mean() * 100, 1),
    "Distancia promedio": round(df_resultados['distancia_top1'].mean(), 4),
    "Distancia mínima": round(df_resultados['distancia_top1'].min(), 4),
    "Distancia máxima": round(df_resultados['distancia_top1'].max(), 4)
}

print("\nResumen estadístico:")
for k, v in resumen.items():
    print(f"  {k}: {v}")

Informe de cobertura guardado en /home/jupyteruser/work/corpus_upeu/metadatos/informe_cobertura.csv

Resumen estadístico:
  Total consultas: 10
  Cubiertas: 10
  No cubiertas: 0
  Tasa cobertura: 100.0
  Distancia promedio: 0.3133
  Distancia mínima: 0.1492
  Distancia máxima: 0.4452
